# 00 — Shared loader and contracts

**Pregunta:** ¿estamos cargando los artifacts correctos y entendiendo sus contratos?

Este notebook no es un reporte financiero. Es el contrato de lectura para el paquete de reportes contables/profesionales.

Responsabilidades:

- localizar el repo y el bundle `public/accounting/latest`;
- verificar artifacts requeridos y opcionales;
- normalizar métricas anuales;
- listar años, monedas, `metric_id`, dimensiones y fuentes;
- detectar problemas de lectura antes de que los reportes narrativos arranquen;
- exportar outputs externos limpios a `out/professional_pack/latest`.

No responsabilidades:

- no recalcula lógica core;
- no clasifica transacciones;
- no decide saldos;
- no inventa caja;
- no suma ARS + USD;
- no cuenta todavía la historia financiera.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 240)

cwd = Path.cwd().resolve()
repo_root = None
for p in [cwd, *cwd.parents]:
    if (p / "Makefile").exists() and (p / "accounting").is_dir():
        repo_root = p
        break
if repo_root is None:
    raise FileNotFoundError("Could not find repo root. Run from inside accounting-backend.")

reports_dir = repo_root / "accounting" / "notebooks" / "accounting_reports"
if str(reports_dir) not in sys.path:
    sys.path.insert(0, str(reports_dir))

from _shared import (
    find_repo_root, professional_pack_dir, inspect_artifacts, load_annual_dashboard_metrics,
    metric_inventory, dimension_inventory, qa_findings, build_readiness_summary,
    export_shared_contract_outputs, DEFAULT_REQUIRED_ARTIFACTS, DEFAULT_OPTIONAL_ARTIFACTS,
)

repo_root = find_repo_root(repo_root)
pack_dir = professional_pack_dir(repo_root)
print("repo_root:", repo_root)
print("professional_pack:", pack_dir)


## 1. Artifact registry

Los artifacts **required** son necesarios para que los demás reportes funcionen. Los **optional** agregan QA, drilldown o contexto, pero no deberían romper el 00 si faltan.

In [ ]:
artifact_inventory = inspect_artifacts(
    repo_root,
    required=DEFAULT_REQUIRED_ARTIFACTS,
    optional=DEFAULT_OPTIONAL_ARTIFACTS,
)

display(artifact_inventory)

missing_required = artifact_inventory[
    artifact_inventory["required"].eq(True) & ~artifact_inventory["exists"].eq(True)
]

if not missing_required.empty:
    display(missing_required)
    raise FileNotFoundError("Missing required artifacts. Run make run-accounting / publish before report notebooks.")


## 2. Load and normalize annual metrics

Fuente principal:

```text
public/accounting/latest/canonical_dashboard/annual_balance_dashboard_metrics.csv
```

La normalización acá es de lectura/reporting: `period` como año string, `Currency` explícita, `value` numérico, y columnas opcionales presentes.

In [ ]:
metrics = load_annual_dashboard_metrics(repo_root)

years = sorted(metrics.loc[metrics["period"].astype(str).str.match(r"^\d{4}$", na=False), "period"].unique().tolist())
currencies = sorted(metrics["Currency"].fillna("N/A").astype(str).unique().tolist())

print("metric rows:", len(metrics))
print("years:", years)
print("currencies:", currencies)
print("metric_ids:", metrics["metric_id"].nunique())

display(metrics.head(30))


## 3. Readiness summary

Resumen corto para saber si el resto del pack puede correr y con qué salvedades.

In [ ]:
metric_inv = metric_inventory(metrics)
dimension_inv = dimension_inventory(metrics)
qa = qa_findings(metrics, artifact_inventory)
readiness = build_readiness_summary(repo_root, metrics, artifact_inventory, qa)

display(readiness)
display(qa)


## 4. Metric inventory

Este inventario es el mapa de qué puede consumir cada notebook. No debe ser interpretado como reporte final; es un índice técnico/humano de métricas disponibles.

In [ ]:
display(metric_inv)

operating_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("IS.")]
funding_distribution_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith(("FUND.", "DIST.", "COV."))]
cash_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith(("BS.CASH", "DQ.CASH"))]
debt_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("ID.DEBT")]
dq_metrics = metric_inv[metric_inv["metric_id"].astype(str).str.startswith("DQ.")]

print("operating metrics:", len(operating_metrics))
print("funding/distribution/coverage metrics:", len(funding_distribution_metrics))
print("cash metrics:", len(cash_metrics))
print("debt metrics:", len(debt_metrics))
print("DQ metrics:", len(dq_metrics))


## 5. Dimension inventory

Este inventario ayuda a detectar cómo vienen las métricas dimensionadas: por propiedad, categoría semántica, actor, debtor/creditor, o `cash_path` cuando esté disponible.

Regla de reporting: si una dimensión es importante para interpretación humana, no debe quedar escondida solo dentro del `line` label; los notebooks posteriores pueden promoverla a columna.

In [ ]:
display(dimension_inv)


## 6. QA checks del shared loader

Estos checks no reemplazan `scripts/check_release.py` ni QA del backend. Son checks de lectura para evitar reportes humanos engañosos.

In [ ]:
display(qa)

available_nan = metrics[
    metrics["value_status"].astype(str).str.lower().eq("available")
    & metrics["value"].isna()
]
if not available_nan.empty:
    print("Rows with value_status=available but NaN value:")
    display(available_nan.head(100))

suspicious_currency = metrics[
    metrics["Currency"].astype(str).isin(["ALL", "Mixed", "ARS+USD", "MULTI", ""])
]
if not suspicious_currency.empty:
    print("Suspicious cross-currency rows:")
    display(suspicious_currency.head(100))


## 7. Professional interpretation guardrails

| Área | Regla |
|---|---|
| Operación | Renta/OPEX/resultado operativo no incluye funding, deuda, dividendos ni gasto personal. |
| Funding | Un aporte no es ingreso operativo. |
| Distribuciones | Retiros, gasto personal y dividendos no son OPEX de propiedad. |
| Deuda | Deuda interna no es OPEX; debe tratarse como stock/flow financiero entre actores. |
| Caja | Si no existe `validated_cash_close` frontend-safe, caja real debe mostrarse como `s/d`. |
| Moneda | ARS y USD pueden convivir como filas con `Currency`, pero nunca se suman. |
| Box | `Box` puede ser órbita de gobernanza, no caja física. |
| Settlement | Pagos directos de inquilinos/actores cancelan obligaciones pero no implican caja PM/FB. |
| QA | `unavailable` debe mostrarse; no equivale a cero. |


## 8. Export shared outputs

Outputs mínimos:

```text
out/professional_pack/latest/shared_contract_inventory.csv
out/professional_pack/latest/shared_contract_summary.md
```

También exporta inventarios auxiliares a `tables/` y `qa/`.

In [ ]:
exported = export_shared_contract_outputs(
    repo_root=repo_root,
    artifact_inventory=artifact_inventory,
    readiness=readiness,
    qa=qa,
    metric_inventory_df=metric_inv,
    dimension_inventory_df=dimension_inv,
)

for key, path in exported.items():
    print(f"{key}: {path.relative_to(repo_root)}")


## 9. Next notebooks enabled by this contract

Propuesta gobernada:

```text
01_balance_dashboard_overview.ipynb
02_cash_and_liquidity.ipynb
03_income_rent_and_operations.ipynb
04_debt_open_items_and_reconciliation.ipynb
05_family_human_storypack.ipynb
```

Cada una debe responder una pregunta humana distinta y exportar outputs propios, sin recalcular lógica core.